In [ ]:
import os
os.environ["PYTHONPATH"] = "/workspace:" + os.environ.get("PYTHONPATH", "")

os.environ["NUPLAN_DATA_ROOT"] = "/workspace/nuplan-mini-sample/nuplan_data"
os.environ["NUPLAN_MAPS_ROOT"] = "/workspace/nuplan-mini-sample/nuplan_data/maps"
os.environ["NUPLAN_DB_FILES"] = "/workspace/nuplan-mini-sample/nuplan_data/nuplan-v1.1/splits/mini/2021.05.12.22.00.38_veh-35_01008_01518.db"

## Pluto notebook inference tutorial

This notebook shows how to run **one forward pass** of PLUTO on a single nuPlan-mini scenario, and how to interpret:

- the **model inputs** (the `feat` / `feat_t.data` feature dictionary)
- the **model outputs** (`candidate_trajectories`, `probability`, `output_trajectory`, `output_prediction`)

Notes:
- The feature builder produces data in the **ego-local frame** at the current timestep (relative to the ego pose). The planner later converts selected trajectories back to global.
- `iteration=0` means “at the first timestep of the scenario”.


In [ ]:
import torch
from src.models.pluto.pluto_model import PlanningModel
from src.planners.ml_planner_utils import load_checkpoint
from src.feature_builders.pluto_feature_builder import PlutoFeatureBuilder

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

feature_builder = PlutoFeatureBuilder(
    radius=120,
    history_horizon=2,
    future_horizon=8,
    sample_interval=0.1,
    max_agents=48,
    build_reference_line=True,
)

model = PlanningModel(
    dim=128,
    state_channel=6,
    polygon_channel=6,
    history_channel=9,
    history_steps=21,
    future_steps=80,
    encoder_depth=4,
    decoder_depth=4,
    drop_path=0.2,
    dropout=0.1,
    num_heads=4,
    num_modes=12,
    state_dropout=0.75,
    use_ego_history=False,
    state_attn_encoder=True,
    use_hidden_proj=True,
    cat_x=True,
    ref_free_traj=True,
    feature_builder=feature_builder,
)

model.load_state_dict(load_checkpoint("/workspace/checkpoints/pluto_1M_aux_cil.ckpt"), strict=True)
model.eval().to(device)

## 1) Load the model + checkpoint

We construct `PlanningModel` with the **same hyperparameters** used by the checkpoint (e.g. `num_modes=12`, `use_hidden_proj=True`, `cat_x=True`, `ref_free_traj=True`).

If these don’t match, `load_state_dict()` will fail with missing/unexpected keys or shape mismatches.


In [ ]:
import os

from nuplan.planning.scenario_builder.nuplan_db.nuplan_scenario_builder import NuPlanScenarioBuilder
from nuplan.planning.scenario_builder.scenario_filter import ScenarioFilter
from nuplan.planning.utils.multithreading.worker_sequential import Sequential

NUPLAN_DATA_ROOT = os.environ["NUPLAN_DATA_ROOT"]
NUPLAN_MAPS_ROOT = os.environ["NUPLAN_MAPS_ROOT"]
NUPLAN_DB_FILES = os.environ["NUPLAN_DB_FILES"]

scenario_builder = NuPlanScenarioBuilder(
    data_root=NUPLAN_DATA_ROOT,
    map_root=NUPLAN_MAPS_ROOT,
    sensor_root=f"{NUPLAN_DATA_ROOT}/nuplan-v1.1/sensor_blobs",
    db_files=[NUPLAN_DB_FILES],
    map_version="nuplan-maps-v1.0",
    include_cameras=False,
    max_workers=1,
    verbose=True,
)

scenario_filter = ScenarioFilter(
    scenario_types=None,
    scenario_tokens=None,
    log_names=None,
    map_names=None,
    num_scenarios_per_type=1,
    limit_total_scenarios=1,
    timestamp_threshold_s=0.0,
    ego_displacement_minimum_m=None,
    expand_scenarios=False,
    remove_invalid_goals=True,
    shuffle=False,
)

worker = Sequential()
scenarios = scenario_builder.get_scenarios(scenario_filter=scenario_filter, worker=worker)
print("num scenarios:", len(scenarios))
scenario = scenarios[0]
print("log:", scenario.log_name, "token:", scenario.token, "type:", scenario.scenario_type)


## 2) Build a `scenario`

A `scenario` is a short driving snippet extracted from the nuPlan DB (log + token), with access to:
- ego past/future states
- tracked objects (agents)
- map API
- traffic lights
- route / mission goal

We use `NuPlanScenarioBuilder` + `ScenarioFilter` to retrieve **exactly 1** scenario from the mini DB.


In [ ]:
# Build model inputs from the scenario and run one forward pass
fb = model.get_list_of_required_feature()[0]
feat = fb.get_features_from_scenario(scenario, iteration=0)
feat_t = feat.collate([feat.to_feature_tensor()]).to_device(device)

with torch.no_grad():
    out = model.forward(feat_t.data)

out.keys(), out["candidate_trajectories"].shape, out["probability"].shape, out["output_prediction"].shape


## 3) Build model input features (`feat`)

`PlutoFeatureBuilder.get_features_from_scenario(...)` returns a `PlutoFeature` object.

Key idea: it builds a nested dict of numpy arrays describing **ego history**, **nearby agents**, **map polylines/polygons**, and optionally a **reference line**.

In code:
- `feat.data` is the raw nested dict (numpy)
- `feat.to_feature_tensor()` converts numpy → torch tensors
- `feat.collate([...])` adds a batch dimension
- `feat_t.data` is what we pass into `model.forward(...)`


### Does the model use history as input?

**Yes.** PLUTO consumes a fixed-length history window for ego + agents.

- **History length** comes from the feature builder settings:
  - `history_horizon=2.0s`, `sample_interval=0.1s` → `history_samples = 20`
  - the feature builder concatenates **past + current** → `history_steps = history_samples + 1 = 21`

- **Where history lives in the input feature dict**:
  - `feat.data["agent"]["position"]` has shape `(A, T, 2)` where the **first 21 steps** are the history window.
  - the ego is inserted as agent index 0, so:
    - ego history: `feat.data["agent"]["position"][0, :21]`
    - other agents history: `feat.data["agent"]["position"][1:, :21]`
  - `feat.data["agent"]["valid_mask"]` marks which timesteps are present per agent.

- **How the model uses it** (see `src/models/pluto/pluto_model.py`):
  - it explicitly slices the history window: `data["agent"]["valid_mask"][..., :history_steps]`
  - it defines the **current state** as the *last history step*: `position[..., history_steps - 1]`
  - it uses the history mask to compute padding / which agents are “present”.


In [ ]:
# Render just this step
from src.feature_builders.nuplan_scenario_render import NuplanScenarioRender
import imageio

renderer = NuplanScenarioRender()
img = renderer.render_from_scenario(
    scenario=scenario,
    iteration=0,
    planning_trajectory=out["output_trajectory"][0].detach().cpu().numpy(),
    candidate_trajectories=out["candidate_trajectories"][0].detach().cpu().numpy().reshape(-1, 80, 3),
    predictions=out["output_prediction"][0].detach().cpu().numpy(),
    return_image=True,
)

imageio.imwrite("/workspace/sim_out_video/step0.png", img)

In [ ]:
# Inspect the feature dict structure (top-level keys + a few shapes)

def _shape(x):
    try:
        return tuple(x.shape)
    except Exception:
        return type(x)

print('feat.data keys:', list(feat.data.keys()))
print('agent.position:', _shape(feat.data['agent']['position']))
print('agent.valid_mask:', _shape(feat.data['agent']['valid_mask']))
print('map.polygon_center:', _shape(feat.data['map']['polygon_center']))
print('reference_line keys:', list(feat.data.get('reference_line', {}).keys()))


## 4) Model outputs (`out`)

`out` is a dictionary of torch tensors.

The most useful keys for inference:
- **`out["candidate_trajectories"]`**: many trajectory hypotheses (multi-modal). Shape is typically `(B, R, M, T, 3)` where:
  - `B`: batch size
  - `R`: number of reference lines (route hypotheses)
  - `M`: number of modes (`num_modes=12` here)
  - `T`: timesteps (`future_steps=80`)
  - `3`: `(x, y, heading)` in the **local** frame
- **`out["probability"]`**: score/logit per candidate, shape `(B, R, M)`
- **`out["output_trajectory"]`**: the single “best” trajectory chosen by `argmax(probability)` after masking invalid reference lines
- **`out["output_prediction"]`**: predicted future states for other agents (used by the rule-based scorer in the full planner)

In the full planner (`src/planners/pluto_planner.py`), these candidates are also evaluated by rule-based checks (collisions, drivable area, lights, etc.), then the best candidate is selected.


In [ ]:
# Inspect output shapes
print('out keys:', list(out.keys()))
print('candidate_trajectories:', tuple(out['candidate_trajectories'].shape))
print('probability:', tuple(out['probability'].shape))
print('output_trajectory:', tuple(out['output_trajectory'].shape))
print('output_prediction:', tuple(out['output_prediction'].shape))

# Best candidate index (matches how PlanningModel picks output_trajectory)
flat = out['probability'].reshape(out['probability'].shape[0], -1)
print('best idx (flattened):', flat.argmax(-1).tolist())
